In [1]:
!pip install deep-translator
!pip install googletrans
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 5.7 MB/s eta 0:00:00


In [137]:
import pandas as pd
import re
import json
import anthropic
from typing import Optional
from pathlib import Path

In [132]:
class TableLoader:
    """Handles table loading and cleaning operations."""

    def load_and_clean(self, csv_path: str) -> pd.DataFrame:
        """
        Load CSV file and clean the data.

        Args:
            csv_path: Path to the CSV file

        Returns:
            Cleaned DataFrame with asterisks removed
        """
        # Read CSV treating all rows equally (no header assumption)
        df = pd.read_csv(csv_path, header=0, dtype=str)

        # Remove asterisks from all cells
        df = df.applymap(lambda x: str(x).replace('*', '') if pd.notna(x) else '')

        return df

In [ ]:
class HardRuleClassifier:
    """Applies hard-coded classification rules to table cells."""

    def __init__(self, threshold: float = 0.8, consistency_threshold: float = 0.3):
        """
        Initialize the classifier with configurable threshold.

        Args:
            threshold: Minimum ratio for row identity rule (default 0.9)
        """
        self.threshold = threshold
        self.consistency_threshold = consistency_threshold

    def classify(self, table_df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply all hard-coded rules to classify cells.

        Args:
            table_df: Input table DataFrame

        Returns:
            Mask DataFrame with "feature", "data-point", or "undecided"
        """
        # Initialize mask with "undecided"
        mask = pd.DataFrame("undecided", index=table_df.index, columns=table_df.columns)

        # Rule 1: Row identity check
        for idx, row in table_df.iterrows():
            if self._check_row_identity(row):
                mask.iloc[idx] = "feature"

        # Rules 2-4: Numeric patterns and missing values
        for i in range(len(table_df)):
            for j in range(len(table_df.columns)):
                # Only check if cell is still undecided
                if mask.iloc[i, j] == "undecided":
                    cell_value = str(table_df.iloc[i, j]).strip()
                    if self._is_numeric_pattern(cell_value):
                        mask.iloc[i, j] = "data-point"

        # Rule 5: Row consistency enforcement
        mask = self._enforce_row_consistency(mask)

        return mask

    def _enforce_row_consistency(self, mask: pd.DataFrame) -> pd.DataFrame:
        """
        If a row is more features or data points, change all cells to that classification,
        given that more than 30% of the row is that classification.

        Args:
            mask: Current mask DataFrame

        Returns:
            Updated mask with row consistency enforced
        """
        for idx, row in mask.iterrows():
            # Count occurrences of each classification
            feature_count = (row == "feature").sum()
            datapoint_count = (row == "data-point").sum()
            total_cells = len(row)

            # Check if features exceed threshold
            if feature_count > datapoint_count:
                if feature_count / total_cells >= self.consistency_threshold:
                    mask.iloc[idx] = "feature"
            # Check if data-points exceed threshold
            elif datapoint_count > feature_count:
                if datapoint_count / total_cells >= self.consistency_threshold:
                    mask.iloc[idx] = "data-point"

        return mask

    def _check_row_identity(self, row: pd.Series) -> bool:
        """
        Check if a row meets the identity threshold.

        Args:
            row: A row from the DataFrame

        Returns:
            True if >= threshold of cells are identical
        """
        # Strip whitespace from all cells
        cleaned_row = row.apply(lambda x: str(x).strip())

        # Count occurrences of the most common value
        if len(cleaned_row) == 0:
            return False

        value_counts = cleaned_row.value_counts()
        if len(value_counts) == 0:
            return False

        most_common_count = value_counts.iloc[0]
        ratio = most_common_count / len(cleaned_row)

        return ratio >= self.threshold

    def _is_numeric_pattern(self, cell: str) -> bool:
        """
        Check if cell matches any numeric pattern or missing value.

        Args:
            cell: Cell value as string

        Returns:
            True if cell matches numeric patterns or missing values
        """
        # Check for missing values
        if cell in ["--", "-"]:
            return True

        # Pattern 1: Decimal numbers (e.g., 123.45)
        if re.match(r'^\d+\.\d+', cell):
            return True

        # Pattern 2: Numbers with thousand separators (e.g., 1,234)
        if re.match(r'^\d{1,3}(,\d{3})+', cell):
            return True

        # Pattern 3: Combined - thousand separators with decimal (e.g., 1,234.56)
        if re.match(r'^\d{1,3}(,\d{3})+\.\d+', cell):
            return True

        return False

In [ ]:
class LLMClassifier:
    """Handles LLM-based classification for undecided cells."""

    # Model constants
    MODEL_HAIKU = "claude-3-5-haiku-latest"
    MODEL_SONNET = "claude-sonnet-4-20250514"

    def __init__(self, api_key: Optional[str] = None):
        """
        Initialize the LLM classifier with API credentials.

        Args:
            api_key: Anthropic API key (uses env var if None)
        """
        if api_key is None:
            import os
            api_key = os.getenv("ANTHROPIC_API_KEY")

        if not api_key:
            raise ValueError("API key must be provided or set in ANTHROPIC_API_KEY environment variable")

        self.client = anthropic.Anthropic(api_key=api_key)

    def _call_llm_with_model(self, prompt: str, model: str) -> str:
        """
        Call the LLM API with a specific model.

        Args:
            prompt: The prompt to send to the LLM
            model: The model identifier to use

        Returns:
            The response text from the LLM

        Raises:
            Exception: If the API call fails
        """
        response = self.client.messages.create(
            model=model,
            max_tokens=5000,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text.strip()



    def _build_prompt(self, table_name: str, table_df: pd.DataFrame, partial_mask: pd.DataFrame) -> str:
        """
        Build the prompt for LLM classification.

        Args:
            table_name: Name of the table
            table_df: Table data
            partial_mask: Current partial mask

        Returns:
            Formatted prompt string
        """
        # Convert DataFrames to lists for cleaner prompt
        table_list = table_df.values.tolist()
        mask_list = partial_mask.values.tolist()
        rows, cols = partial_mask.shape

        # print(f"DEBUG: table_df shape: {table_df.shape}")
        # print(f"DEBUG: partial_mask shape: {partial_mask.shape}")
        # print(f"DEBUG: table_list length: {len(table_list)}")
        # print(f"DEBUG: mask_list length: {len(mask_list)}")

        prompt = f"""
Table name: "{table_name}"

Table content:
{json.dumps(table_list, ensure_ascii=False, indent=2)}

Current partial mask:
{json.dumps(mask_list, ensure_ascii=False, indent=2)}

Task: Classify the "undecided" cells in the partial mask as either "feature" or "data-point".
CRITICAL: Return EXACTLY {rows} rows and {cols} columns in your response.

CLASSIFICATION RULES:
1. Preserve existing classifications - do NOT change cells already marked as "feature" or "data-point"
2. Make your best judgment for "undecided" cells based on context - classify them as either "feature" or "data-point"
3. Only leave a cell as "undecided" if the content is truly ambiguous (e.g., completely empty or meaningless symbols)

CONTEXT CLUES TO USE:
- Look at patterns in the same row and column
- Consider the table name and overall structure
- If a cell contains text that describes or labels something → "feature"
- If a cell contains a specific value, instance, or measurement → "data-point"

Classification definitions:
- "feature": Headers, labels, categories, descriptors, dimensions, attribute names
- "data-point": Actual values, specific instances, measurements, observations, counts

Return ONLY the complete mask as a JSON array of arrays with EXACTLY {rows} rows and {cols} columns.
Do not skip any rows or columns. Do not add any rows or columns.
Every cell must be either "feature", "data-point", or "undecided".
"""
        return prompt

    def classify_undecided(self, table_name: str, table_df: pd.DataFrame, partial_mask: pd.DataFrame, model: str = None) -> pd.DataFrame:
        """
        Classify remaining undecided cells using LLM with hybrid approach.

        Args:
            table_name: Name/title of the table
            table_df: Original table data
            partial_mask: Current mask with some classifications
            model: Deprecated parameter, kept for compatibility

        Returns:
            Updated mask DataFrame
        """
        # Build prompt (same for both models)
        prompt = self._build_prompt(table_name, table_df, partial_mask)

        # First attempt with Haiku
        try:
            response_text = self._call_llm_with_model(prompt, self.MODEL_HAIKU)
            final_mask, success = self._parse_response(response_text, partial_mask)

            if success:
                return final_mask
            else:
                # Shape mismatch detected, try Sonnet
                print("Shape mismatch detected with Haiku. Switching to Sonnet-4...")
                try:
                    response_text = self._call_llm_with_model(prompt, self.MODEL_SONNET)
                    final_mask, success = self._parse_response(response_text, partial_mask)

                    if success:
                        print("Sonnet-4 succeeded in generating correct shape.")
                        return final_mask
                    else:
                        print("Sonnet-4 also failed to generate correct shape. Using partial mask.")
                        return partial_mask

                except Exception as e:
                    print(f"Sonnet-4 API call failed: {e}. Using partial mask.")
                    return partial_mask

        except Exception as e:
            print(f"Haiku API call failed: {e}. Trying Sonnet-4...")
            # If Haiku fails completely, try Sonnet
            try:
                response_text = self._call_llm_with_model(prompt, self.MODEL_SONNET)
                final_mask, success = self._parse_response(response_text, partial_mask)

                if success:
                    print("Sonnet-4 succeeded after Haiku failure.")
                    return final_mask
                else:
                    print("Sonnet-4 failed to generate correct shape. Using partial mask.")
                    return partial_mask

            except Exception as e:
                    print(f"Sonnet-4 API call failed: {e}. Using partial mask.")
                    return partial_mask

    def _parse_response(self, response_text: str, partial_mask: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
        """
        Parse LLM response and update the mask.

        Args:
            response_text: Raw response from LLM
            partial_mask: Current mask to update

        Returns:
            Tuple of (Updated mask DataFrame, Success flag)
        """
        try:
            # Try to extract JSON from response
            # Handle case where LLM might include extra text
            json_start = response_text.find('[')
            json_end = response_text.rfind(']') + 1

            if json_start != -1 and json_end > json_start:
                json_str = response_text[json_start:json_end]
                mask_list = json.loads(json_str)

                # Create new mask DataFrame
                updated_mask = pd.DataFrame(mask_list)

                # Validate dimensions match
                if updated_mask.shape == partial_mask.shape:
                    # Additional validation: preserve already classified cells
                    for i in range(len(partial_mask)):
                        for j in range(len(partial_mask.columns)):
                            original = partial_mask.iloc[i, j]
                            if original in ["feature", "data-point"]:
                                # Preserve already classified cells
                                updated_mask.iloc[i, j] = original
                    return updated_mask, True  # Success
                else:
                    print(f"Dimension mismatch in LLM response: Expected {partial_mask.shape}, got {updated_mask.shape}")
                    return partial_mask, False  # Shape mismatch
            else:
                print("Could not find JSON array in response")
                return partial_mask, False  # Parse failure

        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            return partial_mask, False  # JSON error
        except Exception as e:
            print(f"Error parsing LLM response: {e}")
            return partial_mask, False  # General error

In [140]:
class TableClassifier:
    """Orchestrates the entire table classification process."""

    def __init__(self, api_key: Optional[str] = None):
        """
        Initialize the complete classification system.

        Args:
            api_key: Anthropic API key (uses env var if None)
        """
        self.loader = TableLoader()
        self.hard_classifier = HardRuleClassifier()
        self.llm_classifier = LLMClassifier(api_key)

    def classify_table(self, csv_path: str, table_name: str) -> pd.DataFrame:
        """
        Complete classification pipeline for a table.

        Args:
            csv_path: Path to the CSV file
            table_name: Name/title of the table for context

        Returns:
            Final mask DataFrame with classifications
        """
        # Step 1: Load and clean the table
        table_df = self.loader.load_and_clean(csv_path)
        # print(f"Loaded table shape: {table_df.shape}")
        # print(f"First 2 rows:\n{table_df.head(2)}")
        # print(f"Last 2 rows:\n{table_df.tail(2)}")

        # Step 2: Apply hard-coded rules
        partial_mask = self.hard_classifier.classify(table_df)
        # print(f"Partial mask shape: {partial_mask.shape}")

        # Step 3: Apply LLM classification for undecided cells only if cells are left "undecided"
        if "undecided" in partial_mask.values:
            final_mask = self.llm_classifier.classify_undecided(table_name, table_df, partial_mask)
        else:
            final_mask = partial_mask

        return final_mask

    def load_table_names(self, summary_path: str = "/content/tables_summary.json") -> dict:
        """
        Load table name mappings from the summary JSON file.

        Args:
            summary_path: Path to the tables_summary.json file

        Returns:
            Dictionary mapping CSV identifiers to table names
        """
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"Warning: {summary_path} not found. Using filenames as table names.")
            return {}
        except json.JSONDecodeError as e:
            print(f"Error parsing {summary_path}: {e}. Using filenames as table names.")
            return {}

    def process_single_table(self, input_path: str, output_path: str, table_name: str) -> bool:
        """
        Process a single table and save its classification mask.

        Args:
            input_path: Path to the input CSV file
            output_path: Path where the mask CSV should be saved
            table_name: Human-readable name of the table for LLM context

        Returns:
            True if processing succeeded, False otherwise
        """
        try:
            # Process the table using existing classification pipeline
            mask_df = self.classify_table(input_path, table_name)

            # Save the mask to CSV
            mask_df.to_csv(output_path, index=False)

            return True

        except Exception as e:
            print(f"Failed to process {os.path.basename(input_path)}: {e}")
            return False

    def process_all_tables(self,
                          input_dir: str = "/content/tables",
                          output_dir: str = "/content/masks",
                          summary_path: str = "/content/tables_summary.json") -> None:
        """
        Process all CSV files in the input directory and save masks to output directory.

        Args:
            input_dir: Directory containing input CSV files
            output_dir: Directory where mask CSV files will be saved
            summary_path: Path to the tables_summary.json file
        """
        # Ensure output directory exists
        self.ensure_output_directory(output_dir)

        # Load table name mappings
        table_names = self.load_table_names(summary_path)

        # Get all CSV files in input directory
        input_path = Path(input_dir)
        csv_files = list(input_path.glob("*.csv"))

        if not csv_files:
            print(f"No CSV files found in {input_dir}")
            return

        # Process each CSV file
        for csv_file in csv_files:
            # Get filename without extension for identifier lookup
            identifier = csv_file.stem
            print(identifier)

            # Get table name from mapping or use filename
            table_name = table_names.get(identifier, identifier)

            # Construct output path with same filename
            output_path = Path(output_dir) / csv_file.name

            # Process the table
            self.process_single_table(str(csv_file), str(output_path), table_name)

    def ensure_output_directory(self, output_dir: str) -> None:
        """
        Ensure the output directory exists, create it if necessary.

        Args:
            output_dir: Path to the output directory
        """
        Path(output_dir).mkdir(parents=True, exist_ok=True)

In [143]:
api_key = "API_KEY_HERE" # DELETE WHEN FINISHED
classifier = TableClassifier(api_key=api_key)
classifier.process_all_tables()